# Electrolyte Box Equilibration: NVT -> NPT

Protocol: NVT Langevin 50 ps (thermalization), then NPT 50 ps (density relaxation), 300 K, 1 atm, MACE-MP-0 + D3.

Each box gets its own folder on Google Drive. Only the most recent checkpoint is kept per box, so if the runtime disconnects just resume from the last checkpoint.

---
## 1. Install and import (run once per session)

In [ ]:
# Install deps. torch-dftd has to come from source for MACE dispersion to work.
!pip install -q ase mace-torch
!pip install torch-dftd
!git clone https://github.com/pfnet-research/torch-dftd
!pip install -e /content/torch-dftd

import sys, os
torch_dftd_path = '/content/torch-dftd'
if os.path.exists(torch_dftd_path) and torch_dftd_path not in sys.path:
    sys.path.insert(0, torch_dftd_path)
    print(f"Added {torch_dftd_path} to sys.path")

In [ ]:
import os, sys, glob, json, shutil
import numpy as np
import matplotlib.pyplot as plt
from ase.io import read, write
from ase.md.langevin import Langevin
from ase.md.npt import NPT
from ase.md.velocitydistribution import MaxwellBoltzmannDistribution
from ase import units
from mace.calculators import mace_mp
import torch
from torch_dftd.torch_dftd3_calculator import TorchDFTD3Calculator

print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
# mount Drive so checkpoints survive a runtime disconnect
from google.colab import drive
drive.mount('/content/gdrive')

CHECKPOINT_ROOT = '/content/gdrive/MyDrive/electrolyte_equilibration_checkpoints'
os.makedirs(CHECKPOINT_ROOT, exist_ok=True)
print(f"Checkpoint root: {CHECKPOINT_ROOT}")

In [ ]:
def get_calculator(device=None):
    if device is None:
        device = "cuda" if torch.cuda.is_available() else "cpu"
    return mace_mp(model="medium", dispersion=True, device=device,
                   default_dtype="float32")


def get_box_dir(box_name):
    d = os.path.join(CHECKPOINT_ROOT, box_name)
    os.makedirs(d, exist_ok=True)
    return d


def save_checkpoint(atoms, box_name, step, phase, temps, pes):
    """Saves a checkpoint to Drive, clearing out older ones for this box so only the latest sticks around."""
    box_dir = get_box_dir(box_name)

    for old in glob.glob(os.path.join(box_dir, "checkpoint_*.xyz")):
        os.remove(old)
    for old in glob.glob(os.path.join(box_dir, "checkpoint_*.npz")):
        os.remove(old)

    xyz_path = os.path.join(box_dir, f"checkpoint_{phase}_step{step}.xyz")
    write(xyz_path, atoms, format="extxyz")

    npz_path = os.path.join(box_dir, f"checkpoint_{phase}_step{step}.npz")
    np.savez(npz_path, temps=np.array(temps), pes=np.array(pes),
             step=step, phase=phase)

    print(f"    [SAVE] {phase} step {step} -> {box_dir} (older checkpoints deleted)")


def load_checkpoint(box_name):
    """Loads the most recent checkpoint for a box, or (None, 0, None, [], []) if there isn't one."""
    box_dir = get_box_dir(box_name)
    xyz_files = sorted(glob.glob(os.path.join(box_dir, "checkpoint_*.xyz")))
    npz_files = sorted(glob.glob(os.path.join(box_dir, "checkpoint_*.npz")))

    if not xyz_files or not npz_files:
        return None, 0, None, [], []

    atoms = read(xyz_files[-1])
    data = np.load(npz_files[-1], allow_pickle=True)
    step = int(data["step"])
    phase = str(data["phase"])
    temps = data["temps"].tolist()
    pes = data["pes"].tolist()

    return atoms, step, phase, temps, pes


def run_md_phase(atoms, box_name, phase, n_steps, start_step=0,
                 dt_fs=1.0, T=300.0, friction=0.01, pressure_GPa=None,
                 log_interval=100, checkpoint_interval=5000,
                 existing_temps=None, existing_pes=None):
    """Runs NVT or NPT with periodic checkpointing to Drive (only the latest checkpoint per box is kept)."""
    atoms.calc = get_calculator()

    if phase == "npt" and pressure_GPa is not None:
        pressure_eV_A3 = pressure_GPa * 0.006242
        dyn = NPT(atoms, timestep=dt_fs * units.fs, temperature_K=T,
                  externalstress=pressure_eV_A3,
                  ttime=25 * units.fs, pfactor=None)
    else:
        dyn = Langevin(atoms, timestep=dt_fs * units.fs, temperature_K=T,
                       friction=friction / units.fs)

    dyn.nsteps = start_step
    temps = list(existing_temps) if existing_temps else []
    pes = list(existing_pes) if existing_pes else []
    remaining = n_steps - start_step

    print(f"  Phase: {phase.upper()}, steps {start_step} -> {n_steps} ({remaining} remaining)")

    def logger():
        t = atoms.get_temperature()
        pe = atoms.get_potential_energy() / len(atoms)
        temps.append(t)
        pes.append(pe)
        if dyn.nsteps % 1000 == 0:
            vol = atoms.get_volume()
            density = sum(atoms.get_masses()) / vol * 1.66054
            print(f"    step {dyn.nsteps:6d}/{n_steps}  T={t:.1f} K  "
                  f"PE={pe:.4f} eV/atom  rho={density:.3f} g/cm3")

    def checkpointer():
        if dyn.nsteps > start_step and dyn.nsteps % checkpoint_interval == 0:
            save_checkpoint(atoms, box_name, dyn.nsteps, phase, temps, pes)

    dyn.attach(logger, interval=log_interval)
    dyn.attach(checkpointer, interval=log_interval)

    if remaining <= 0:
        print(f"  Already complete at step {start_step}.")
        return atoms, np.array(temps), np.array(pes)

    dyn.run(remaining)
    save_checkpoint(atoms, box_name, n_steps, phase, temps, pes)

    return atoms, np.array(temps), np.array(pes)


def save_final(atoms, box_name):
    box_dir = get_box_dir(box_name)
    path = os.path.join(box_dir, f"{box_name}.xyz")
    write(path, atoms, format="extxyz")
    has_nan = np.any(np.isnan(atoms.get_positions()))
    if has_nan:
        print(f"  WARNING: NaN in final positions for {box_name}!")
    else:
        print(f"  Final structure: {path}")
    return path


def plot_diagnostics(box_name, temps, pes):
    if len(temps) == 0:
        print("No data to plot.")
        return
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4))
    time_ps = np.arange(len(temps)) * 0.1  # log_interval=100, dt=1fs -> 0.1 ps
    w = min(50, max(1, len(temps) // 4))
    ax1.plot(time_ps, temps, alpha=0.4, lw=0.5)
    if w > 1:
        ax1.plot(time_ps[w-1:], np.convolve(temps, np.ones(w)/w, 'valid'), 'r', lw=1.5)
    ax1.axhline(300, color='k', ls='--', alpha=0.5)
    ax1.set(xlabel='Time (ps)', ylabel='Temperature (K)', title=f'{box_name}: Temperature')
    ax2.plot(time_ps, pes, alpha=0.4, lw=0.5)
    if w > 1:
        ax2.plot(time_ps[w-1:], np.convolve(pes, np.ones(w)/w, 'valid'), 'r', lw=1.5)
    ax2.set(xlabel='Time (ps)', ylabel='PE (eV/atom)', title=f'{box_name}: Potential Energy')
    plt.tight_layout()
    plt.savefig(os.path.join(get_box_dir(box_name), 'diagnostics.png'), dpi=150)
    plt.show()

print("Functions loaded.")

---
## 2. NVT thermalization

Runs NVT Langevin at 300 K to thermalize velocities and relax local structure. Has to finish before NPT (section 3).

### 2a. Start NVT from scratch

Upload a FIRE-optimized `.xyz` file (from `data/colab_upload/`).  
The box name is derived from the filename automatically.

In [ ]:
from google.colab import files

print("Select the FIRE-optimized .xyz file to start NVT:")
uploaded = files.upload()

input_file = list(uploaded.keys())[0]
box_name = input_file.replace("_opt.xyz", "").replace(".xyz", "")
print(f"\nInput file: {input_file}")
print(f"Box name:   {box_name}")
print(f"Checkpoints: {CHECKPOINT_ROOT}/{box_name}/")

In [ ]:
NVT_STEPS = 50000  # 50 ps

atoms = read(input_file)
atoms.pbc = True
print(f"Loaded: {len(atoms)} atoms, cell = {atoms.cell.lengths()}")

MaxwellBoltzmannDistribution(atoms, temperature_K=300.0)

print(f"\n{'='*60}")
print(f"  NVT thermalization: {box_name}")
print(f"{'='*60}")
atoms, temps, pes = run_md_phase(atoms, box_name, phase="nvt", n_steps=NVT_STEPS)

final_path = save_final(atoms, box_name)
plot_diagnostics(box_name, temps, pes)
print(f"\nNVT complete. Final structure: {final_path}")

### 2b. Resume NVT from checkpoint

If runtime disconnected during NVT, use this cell to resume.  
It finds the latest checkpoint on Drive and continues from there.

In [ ]:
# List all boxes with checkpoints
print("Boxes with checkpoints on Drive:\n")
box_dirs = sorted([
    d for d in os.listdir(CHECKPOINT_ROOT)
    if os.path.isdir(os.path.join(CHECKPOINT_ROOT, d))
])
for i, d in enumerate(box_dirs):
    box_dir = os.path.join(CHECKPOINT_ROOT, d)
    ckpts = sorted(glob.glob(os.path.join(box_dir, "checkpoint_*.xyz")))
    final = os.path.exists(os.path.join(box_dir, f"{d}.xyz"))
    if ckpts:
        status = "FINISHED" if final else f"latest: {os.path.basename(ckpts[-1])}"
    else:
        status = "no checkpoints"
    print(f"  [{i}] {d:<50s} {status}")

print("\nSet RESUME_INDEX below to the number of the box to resume.")

In [ ]:
RESUME_INDEX = 0  # <-- change this
NVT_STEPS = 50000

resume_name = box_dirs[RESUME_INDEX]
atoms, step, phase, prev_temps, prev_pes = load_checkpoint(resume_name)

if atoms is None:
    print(f"No checkpoint found for {resume_name}. Use 'Start from scratch' (2a) instead.")
else:
    print(f"Resuming: {resume_name}")
    print(f"  Phase: {phase}, Step: {step}, Atoms: {len(atoms)}")
    atoms.pbc = True

    if phase == "nvt" and step < NVT_STEPS:
        atoms, temps, pes = run_md_phase(
            atoms, resume_name, phase="nvt", n_steps=NVT_STEPS,
            start_step=step, existing_temps=prev_temps, existing_pes=prev_pes)
        final_path = save_final(atoms, resume_name)
        plot_diagnostics(resume_name, temps, pes)
        print(f"NVT complete. Final structure: {final_path}")
    elif phase == "nvt" and step >= NVT_STEPS:
        print(f"NVT already complete at step {step}. Proceed to NPT (Section 3).")
    elif phase == "npt":
        print(f"This box is already in NPT phase. Use Section 3b to resume NPT.")

---
## 3. NPT density equilibration

Runs NPT at 300 K, 1 atm to relax the box volume to the correct density. Requires NVT to be complete first (section 2).

### 3a. Start NPT from a completed NVT box

Upload the NVT-equilibrated `.xyz` file, or point to a completed NVT box on Drive.

In [ ]:
from google.colab import files

print("Select the NVT-equilibrated .xyz file to start NPT:")
print("(This can be a file from Drive or a fresh upload)\n")
uploaded = files.upload()

input_file = list(uploaded.keys())[0]
box_name = input_file.replace("_nvt.xyz", "").replace(".xyz", "")
print(f"\nInput file: {input_file}")
print(f"Box name:   {box_name}")
print(f"Checkpoints: {CHECKPOINT_ROOT}/{box_name}/")

In [ ]:
NPT_STEPS = 50000  # 50 ps

atoms = read(input_file)
atoms.pbc = True
print(f"Loaded: {len(atoms)} atoms, cell = {atoms.cell.lengths()}")

print(f"\n{'='*60}")
print(f"  NPT density equilibration: {box_name}")
print(f"{'='*60}")
atoms, temps, pes = run_md_phase(
    atoms, box_name, phase="npt", n_steps=NPT_STEPS,
    pressure_GPa=0.000101325)  # 1 atm

final_path = save_final(atoms, box_name)
plot_diagnostics(box_name, temps, pes)
print(f"\nNPT complete. Final structure: {final_path}")

### 3b. Resume NPT from checkpoint

If runtime disconnected during NPT, use this cell to resume.

In [ ]:
# List all boxes with checkpoints
print("Boxes with checkpoints on Drive:\n")
box_dirs = sorted([
    d for d in os.listdir(CHECKPOINT_ROOT)
    if os.path.isdir(os.path.join(CHECKPOINT_ROOT, d))
])
for i, d in enumerate(box_dirs):
    box_dir = os.path.join(CHECKPOINT_ROOT, d)
    ckpts = sorted(glob.glob(os.path.join(box_dir, "checkpoint_*.xyz")))
    final = os.path.exists(os.path.join(box_dir, f"{d}.xyz"))
    if ckpts:
        status = "FINISHED" if final else f"latest: {os.path.basename(ckpts[-1])}"
    else:
        status = "no checkpoints"
    print(f"  [{i}] {d:<50s} {status}")

print("\nSet RESUME_INDEX below to the number of the box to resume.")

In [ ]:
RESUME_INDEX = 0  # <-- change this
NPT_STEPS = 50000

resume_name = box_dirs[RESUME_INDEX]
atoms, step, phase, prev_temps, prev_pes = load_checkpoint(resume_name)

if atoms is None:
    print(f"No checkpoint found for {resume_name}.")
else:
    print(f"Resuming: {resume_name}")
    print(f"  Phase: {phase}, Step: {step}, Atoms: {len(atoms)}")
    atoms.pbc = True

    if phase == "npt" and step < NPT_STEPS:
        atoms, temps, pes = run_md_phase(
            atoms, resume_name, phase="npt", n_steps=NPT_STEPS,
            start_step=step, pressure_GPa=0.000101325,
            existing_temps=prev_temps, existing_pes=prev_pes)
        final_path = save_final(atoms, resume_name)
        plot_diagnostics(resume_name, temps, pes)
        print(f"NPT complete. Final structure: {final_path}")
    elif phase == "npt" and step >= NPT_STEPS:
        print(f"NPT already complete at step {step}.")
    elif phase == "nvt":
        print(f"This box is still in NVT phase. Use Section 2b to finish NVT first.")

---
## 4. Status check for all boxes

In [ ]:
print(f"{'Box':<50s} {'Status':<20s} {'Details'}")
print("-" * 100)
for d in sorted(os.listdir(CHECKPOINT_ROOT)):
    box_dir = os.path.join(CHECKPOINT_ROOT, d)
    if not os.path.isdir(box_dir):
        continue
    final = os.path.join(box_dir, f"{d}.xyz")
    if os.path.exists(final):
        a = read(final)
        has_nan = np.any(np.isnan(a.get_positions()))
        status = "NaN!" if has_nan else "DONE"
        details = f"{len(a)} atoms"
    else:
        ckpts = sorted(glob.glob(os.path.join(box_dir, "checkpoint_*.npz")))
        if ckpts:
            data = np.load(ckpts[-1], allow_pickle=True)
            status = f"{str(data['phase'])} step {int(data['step'])}"
            details = os.path.basename(ckpts[-1])
        else:
            status = "empty"
            details = ""
    print(f"{d:<50s} {status:<20s} {details}")